## Cell 1 — imports and raw peek:



In [1]:
import pandas as pd

PATH = "../data/raw/AC2025_AnnualisedEntryExit_public.xlsx"

# header=None so pandas doesn't guess — we want to see the raw layout ourselves first
raw = pd.read_excel(PATH, sheet_name="AC2025_AnnualisedEntryExit", header=None, nrows=8)
raw

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18
0,NaN,COUNTS (LU/LO/DLR/TfL Rail)--2025 // Station a...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,(C) Copyright Transport for London 2026,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,Counts by Station,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,version_v08 2026-05-13,NaN,NaN,NaN,NaN,Typical,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,Mode,MNLC,MASC,Station,Coverage,Source,Monday,Midweek (Tue-Thu),Friday,Saturday,Sunday,Monday,Midweek (Tue-Thu),Friday,Saturday,Sunday,Weekly,12-week,Annualised
6,NaN,NaN,NaN,NaN,NaN,NaN,Entry,Entry,Entry,Entry,Entry,Exit,Exit,Exit,Exit,Exit,En/Ex,En/Ex,En/Ex
7,LU,500,ACTu,Acton Town,Station entry/exit,taps,7102.571429,7878.666667,7755.2,5944.5,4493.75,7228.0,8004.833333,7823.4,6212.75,4674.75,98885.421429,1136779.981649,4613379.108771


## Cell 2 — load it properly:

In [2]:
df = pd.read_excel(PATH, sheet_name="AC2025_AnnualisedEntryExit", header=5)
print(df.shape)
df.head()

(470, 19)


,Mode,MNLC,MASC,Station,Coverage,Source,Monday,Midweek (Tue-Thu),Friday,Saturday,Sunday,Monday.1,Midweek (Tue-Thu).1,Friday.1,Saturday.1,Sunday.1,Weekly,12-week,Annualised
0,NaN,NaN,NaN,NaN,NaN,NaN,Entry,Entry,Entry,Entry,Entry,Exit,Exit,Exit,Exit,Exit,En/Ex,En/Ex,En/Ex
1,LU,500.0,ACTu,Acton Town,Station entry/exit,taps,7102.571429,7878.666667,7755.2,5944.5,4493.75,7228.0,8004.833333,7823.4,6212.75,4674.75,98885.421429,1136779.981649,4613379.108771
2,LU,502.0,ALDu,Aldgate,Station entry/exit,taps,10241.0,12544.2,9006.25,5965.5,4651.4,11675.833333,14425.8,11675.25,7933.25,5423.4,147481.883333,1695441.55455,6880587.953844
3,LU,503.0,ALEu,Aldgate East,Station entry/exit,taps,17385.714286,20020.5,18113.8,17370.4,13302.0,16194.714286,19654.625,18311.0,17313.6,11733.333333,248749.936905,2859612.110911,11605125.868361
4,LU,505.0,ALPu,Alperton,Station entry/exit,taps,3692,3476.333333,3600.8,2553.333333,1932.4,3889.333333,3761.666667,3743.0,2545.333333,1875.8,45546,523593.673326,2124893.253753


## Cell 3 — drop the phantom header row and the real footer/totals row:

In [3]:
df = df[df['Station'].notna()].copy()
print(df.shape)

(468, 19)


## Cell 4 — look at the Coverage column, this is where the real quirks live:

In [4]:
df['Coverage'].value_counts(dropna=False)

Coverage
Station entry/exit      419
---see LU---             35
Boarding/alighting       11
---see LO---              1
---see EZL---             1
---station closed---      1
Name: count, dtype: int64

## Cell 5 — apply the cleaning rule, keep only standard entry/exit rows:

In [5]:
clean = df[df['Coverage'] == 'Station entry/exit'].copy()
print(clean.shape)

(419, 19)


## Cell 6 — handle the stations with no 2025 data at all:

In [6]:
missing = clean[clean['Annualised'].apply(lambda x: isinstance(x, str))]
print("Excluded, no 2025 data:", missing['Station'].tolist())

clean = clean[~clean['Annualised'].apply(lambda x: isinstance(x, str))].copy()
clean['Annualised'] = clean['Annualised'].astype(float)
print(clean.shape)

Excluded, no 2025 data: ['Star Lane', 'Tower Gateway', 'Acton Main Line', 'Custom House EL', 'Langley']
(414, 19)


## Cell 7 — sanity check, no double-counting left:

In [7]:
print("Duplicate station names remaining:", clean['Station'].duplicated().sum())

Duplicate station names remaining: 0


## Cell 8 — save the cleaned data so tomorrow's EDA notebook can just load it directly:

In [8]:
clean.to_csv("../data/processed/stations_2025_clean.csv", index=False)
print("Saved:", clean.shape)

Saved: (414, 19)
